## Step 5 : Do the rankings hold?

Reads the 3 ball tables from steps 1 to 3 and the two ranked lists from step 4 (`outputs/04_batters.csv`, `outputs/04_bowlers.csv`).
Writes `outputs/05_batters.csv` and `outputs/05_bowlers.csv` (the final lists, ranked in IPL terms, with how sure I am of every rank),
`outputs/05_splithalf.csv`, `outputs/05_blend.csv`, `outputs/05_parts.csv`, `outputs/05_sensitivity.csv` and `outputs/05_forward.csv` (the tables behind the report).

Step 4 ranked both lists on level. This step tries to break that ranking in 6 ways:
1. Split-half : does one half of the matches predict the other half? For level, for clutch, for a blend of the two, and for the runs and wicket parts of a value
2. IPL against T20I : do the two competitions agree about a player, and is a T20I ball easier?
3. Rule index : does the list change if pressure is the simple rule from step 2?
4. Eligibility : does the list change with other minimums?
5. My own choices : the price of a wicket, the years used, and the shrinking
6. Looking forward : what did the years up to 2025 say about 2026?

Then the final lists : the level in IPL terms (the T20I gap from test 2 taken off), and how often each player is in the top 10 when the matches are drawn again 1,000 times.

In [1]:
# Importing Libraries
import os
import numpy as np
import pandas as pd

output_folder = os.path.join("..", "outputs")
keys = ['match_id', 'innings', 'innings_ball_num']

# Loading Dataset (the 3 tables from steps 1 to 3, one row per ball)
balls = pd.read_parquet(os.path.join(output_folder, "balls_clean.parquet"), columns=keys + [
    'striker_id', 'striker_name', 'bowler_id', 'bowler_name', 'legal', 'comp_group', 'match_date',
    'bat_runs', 'bowler_runs', 'striker_out', 'bowler_wicket'])
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_pressure.parquet"), columns=keys + ['pressure', 'rule_index']), on=keys)
balls = balls.merge(pd.read_parquet(os.path.join(output_folder, "balls_value.parquet"), columns=keys + [
    'batter_value', 'bowler_value', 'wicket_cost', 'exp_bat_runs', 'exp_bat_out', 'exp_bowl_runs', 'exp_bowl_wicket']), on=keys)

# Same as step 4 : Other T20 players are not ranked, a high pressure ball has pressure 80 or more
balls = balls[balls['comp_group'] != 'Other T20']
balls['high'] = balls['pressure'] >= 80
balls['year'] = pd.to_datetime(balls['match_date']).dt.year
year_of_match = balls.groupby('match_id')['year'].first()

# For test 3 : the top fifth of balls again, but picked with the simple rule index from step 2
balls['high_rule'] = balls['rule_index'] >= balls['rule_index'].quantile(0.8)
print(balls.shape)
print("share of high pressure balls:", round(balls['high'].mean(), 3), "with the rule index:", round(balls['high_rule'].mean(), 3),
      "balls that are high in both:", round((balls['high'] & balls['high_rule']).mean(), 3))

# The two ranked lists from step 4 (index = player id), and a name for every player id
step4 = {'batters': pd.read_csv(os.path.join(output_folder, "04_batters.csv"), index_col=0),
         'bowlers': pd.read_csv(os.path.join(output_folder, "04_bowlers.csv"), index_col=0)}
names = {'batters': balls.groupby('striker_id')['striker_name'].first(),
         'bowlers': balls.groupby('bowler_id')['bowler_name'].first()}

(193988, 26)
share of high pressure balls: 0.198 with the rule index: 0.2 balls that are high in both: 0.071


In [2]:
# One row per player per match, with every sum the tests need.
# Every test below is a different way of adding these rows up (one half of the matches, IPL only,
# matches drawn again with repeats), so no test has to go back to the 194,000 balls.
def match_rows(source, player, value):
    source = source.assign(square=source[value] ** 2, high_value=source[value] * source['high'],
                           rule_value=source[value] * source['high_rule'])
    rows = source.groupby([player, 'match_id']).agg(
        balls=(value, 'size'), total=(value, 'sum'), squares=('square', 'sum'),
        high_balls=('high', 'sum'), high_total=('high_value', 'sum'),
        rule_balls=('high_rule', 'sum'), rule_total=('rule_value', 'sum'),
        comp_group=('comp_group', 'first')).reset_index()
    ids = np.sort(rows[player].unique())
    rows['code'] = np.searchsorted(ids, rows[player])
    rows['match_code'] = rows['match_id'].astype('category').cat.codes
    return rows, ids


# Adds the rows up to one line per player. weight = how many times each row counts (once if not given).
# The lines are in the order of the player codes, so line 0 is the player with code 0.
def player_numbers(rows, players, weight=None):
    weight = np.ones(len(rows)) if weight is None else weight
    code = rows['code'].values
    table = pd.DataFrame({column: np.bincount(code, weight * rows[column].values, minlength=players) for column in [
        'balls', 'total', 'squares', 'high_balls', 'high_total', 'rule_balls', 'rule_total']})
    other_balls = table['balls'] - table['high_balls']
    table['raw_level'] = table['total'] / table['balls']
    table['raw_high'] = table['high_total'] / table['high_balls']
    table['delta'] = table['raw_high'] - (table['total'] - table['high_total']) / other_balls

    # Pressure match = 6 or more high pressure balls in the match. beat_average = share of them above the average player
    pressure_match = rows['high_balls'].values >= 6
    table['pressure_matches'] = np.bincount(code, weight * pressure_match, minlength=players)
    table['beat_average'] = np.bincount(code, weight * (pressure_match & (rows['high_total'].values > 0)), minlength=players) / table['pressure_matches']
    return table


# The shrunk level of step 4, from the match rows (same formula : pull the raw level towards the pool average
# by as much as it could be luck, luck = ball by ball luck times the factor for balls of one match moving together)
def shrunk_level(rows, table, in_pool, weight=None):
    weight = np.ones(len(rows)) if weight is None else weight
    code = rows['code'].values
    balls, raw = table['balls'].values, table['raw_level'].values
    in_pool = in_pool & (balls > 0)
    ball_spread = (table['squares'].sum() - (table['total'] ** 2 / table['balls']).sum()) / (balls.sum() - 1)
    luck = ball_spread / np.where(balls > 0, balls, np.nan)
    in_match = (rows['total'].values - rows['balls'].values * raw[code]) / balls[code]
    by_match = np.bincount(code, weight * in_match ** 2, minlength=len(table))
    luck = luck * by_match[in_pool].mean() / luck[in_pool].mean()
    spread = max(raw[in_pool].var(ddof=1) - luck[in_pool].mean(), 0)
    centre = raw[in_pool].mean()
    return centre + (raw - centre) * spread / (spread + luck)


# The level of a player is the sum of the shrunk level of his parts.
# A batter has one part (his value). A bowler has two : runs saved and wickets, shrunk apart and then added (step 4).
# parts = a list of match row tables with the same rows in the same order
def shrunk_parts(parts, players, in_pool, weight=None):
    return sum(shrunk_level(rows, player_numbers(rows, players, weight), in_pool, weight) for rows in parts)


# Who is ranked : the step 4 rules. min_high high pressure balls and min_matches pressure matches,
# and half of both in the IPL itself. The pool for shrinking is everyone with min_high high pressure balls.
def who_is_ranked(rows, players, min_high, min_matches=10):
    everything = player_numbers(rows, players)
    in_ipl = player_numbers(rows[rows['comp_group'] == 'IPL'], players)
    in_pool = (everything['high_balls'] >= min_high).values
    eligible = in_pool & (everything['pressure_matches'] >= min_matches).values \
        & (in_ipl['high_balls'] >= min_high / 2).values & (in_ipl['pressure_matches'] >= min_matches / 2).values
    return in_pool, eligible


# IPL terms. A T20I ball is easier than an IPL ball (test 2 shows it), and the lists are for an IPL auction.
# gap = how much better the same player is per ball in T20Is than in the IPL (pool players with 300+ balls in both,
# a player counts for more if both his numbers have many balls behind them).
# The gap is taken off every T20I ball, so a T20I ball counts for what it is worth in the IPL.
def t20i_gap(rows, players, in_pool):
    ipl = player_numbers(rows[rows['comp_group'] == 'IPL'], players)
    t20i = player_numbers(rows[rows['comp_group'] == 'T20I'], players)
    both = in_pool & (ipl['balls'] >= 300).values & (t20i['balls'] >= 300).values
    gaps = (t20i['raw_level'] - ipl['raw_level'])[both]
    return np.average(gaps, weights=(1 / (1 / ipl['balls'] + 1 / t20i['balls']))[both]), gaps

def in_ipl_terms(rows, gap):
    is_t20i = rows['comp_group'] == 'T20I'
    return rows.assign(total=rows['total'] - gap * rows['balls'] * is_t20i,
                       squares=rows['squares'] + (gap ** 2 * rows['balls'] - 2 * gap * rows['total']) * is_t20i)

# The level of every ranked player in IPL terms, best first. gaps = one T20I gap for every part.
def final_level(parts, ids, in_pool, eligible, gaps):
    parts = [in_ipl_terms(rows, gap) for rows, gap in zip(parts, gaps)]
    level = pd.Series(shrunk_parts(parts, len(ids), in_pool), index=ids)
    return level[eligible & (player_numbers(parts[0], len(ids))['balls'] > 0).values].sort_values(ascending=False)


# Batters on legal balls, bowlers on every delivery (as in step 4)
balls['bowler_runs_part'] = balls['exp_bowl_runs'] - balls['bowler_runs']
balls['bowler_wicket_part'] = balls['wicket_cost'] * (balls['bowler_wicket'].astype(float) - balls['exp_bowl_wicket'])
groups = {}
for label, source, player, value, part_values, min_high in [
    ('batters', balls[balls['legal']], 'striker_id', 'batter_value', ['batter_value'], 100),
    ('bowlers', balls, 'bowler_id', 'bowler_value', ['bowler_runs_part', 'bowler_wicket_part'], 120),
]:
    rows, ids = match_rows(source, player, value)
    parts = [match_rows(source, player, part)[0] for part in part_values]
    in_pool, eligible = who_is_ranked(rows, len(ids), min_high)
    table = player_numbers(rows, len(ids))
    table['level'] = shrunk_parts(parts, len(ids), in_pool)
    gaps = [t20i_gap(part, len(ids), in_pool)[0] for part in parts]
    level = final_level(parts, ids, in_pool, eligible, gaps)
    groups[label] = {'rows': rows, 'parts': parts, 'ids': ids, 'min_high': min_high, 'in_pool': in_pool, 'eligible': eligible, 'table': table,
                     'source': source, 'player': player, 'value': value, 'part_values': part_values,
                     'gaps': gaps, 'ipl_parts': [in_ipl_terms(part, gap) for part, gap in zip(parts, gaps)],
                     'level': level, 'mine': list(level.index[:10])}

    # Check : the numbers built from the match rows must be the step 4 numbers
    mine = table[eligible].set_index(ids[eligible])
    old = step4[label]
    print(f"{label}: {len(rows)} match rows, {len(ids)} players, {in_pool.sum()} in the pool, {eligible.sum()} ranked. "
          f"Same players as step 4: {set(mine.index) == set(old.index)}, "
          f"biggest gap in level {(mine['level'] - old['level']).abs().max():.5f}, in consistency {(mine['beat_average'] - old['beat_average']).abs().max():.5f}, "
          f"same top 10: {list(mine['level'].sort_values(ascending=False).index[:10]) == list(old.index[:10])}")
    print(f"   T20I gap of every part {[round(gap, 3) for gap in gaps]}, together {sum(gaps):+.3f}")
    print(f"   final top 10 (IPL terms): {names[label][groups[label]['mine']].tolist()}")

# 'mine' in every test below is this final top 10

batters: 13229 match rows, 824 players, 121 in the pool, 57 ranked. Same players as step 4: True, biggest gap in level 0.00005, in consistency 0.00005, same top 10: True
   T20I gap of every part [np.float64(0.084)], together +0.084
   final top 10 (IPL terms): ['Abhishek Sharma', 'Tim David', 'Suryakumar Yadav', 'Heinrich Klaasen', 'Rajat Patidar', 'Phil Salt', 'Sai Sudharsan', 'Mitchell Marsh', 'Nicholas Pooran', 'Jos Buttler']
bowlers: 9730 match rows, 596 players, 102 in the pool, 62 ranked. Same players as step 4: True, biggest gap in level 0.00005, in consistency 0.00005, same top 10: True
   T20I gap of every part [np.float64(-0.001), np.float64(0.045)], together +0.043
   final top 10 (IPL terms): ['Jasprit Bumrah', 'Sunil Narine', 'Nathan Ellis', 'Matheesha Pathirana', 'Mustafizur Rahman', 'Josh Hazlewood', 'Bhuvneshwar Kumar', 'Varun Chakravarthy', 'Rashid Khan', 'Mitchell Santner']


In [3]:
# Test 1 : split-half
# I split the matches into two random halves, 100 times. In every split each ranked player is measured in
# both halves, and I take the correlation between the halves over the ranked players.
#   level               : value per ball over all his balls
#   high pressure value : value per ball on his high pressure balls only
#   clutch              : high pressure balls minus his other balls. If it is a skill, it must come back in the other half
#   consistency         : share of his pressure matches above the average player
# Then the question that decides the ranking : what predicts the high pressure value of the OTHER half best?
# score = w * level + (1 - w) * high pressure value, for w from 0 (only high pressure balls) to 1 (only level)
splitter = np.random.RandomState(5)
blend_weights = np.round(np.arange(0, 1.01, 0.1), 1)
same_table, blend_table = {}, {}
for label, group in groups.items():
    rows, players, eligible = group['rows'], len(group['ids']), group['eligible']
    rule_rows = rows.assign(high_balls=rows['rule_balls'], high_total=rows['rule_total'])
    match_code = rows['match_code'].values
    same, blend = [], []
    for run in range(100):
        first = (splitter.rand(match_code.max() + 1) < 0.5)[match_code]
        halves = [player_numbers(rows[side], players)[eligible] for side in [first, ~first]]
        rule_halves = [player_numbers(rule_rows[side], players)[eligible] for side in [first, ~first]]
        same.append({
            'level': halves[0]['raw_level'].corr(halves[1]['raw_level']),
            'high pressure value': halves[0]['raw_high'].corr(halves[1]['raw_high']),
            'clutch': halves[0]['delta'].corr(halves[1]['delta']),
            'clutch (rule index)': rule_halves[0]['delta'].corr(rule_halves[1]['delta']),
            'consistency': halves[0]['beat_average'].corr(halves[1]['beat_average']),
        })
        for one, other in [(0, 1), (1, 0)]:
            blend.append([(w * halves[one]['raw_level'] + (1 - w) * halves[one]['raw_high']).corr(halves[other]['raw_high'])
                          for w in blend_weights])
    same, blend = pd.DataFrame(same), np.array(blend)
    same_table[label + ' average'] = same.mean()
    same_table[label + ' spread'] = same.std()
    same_table[label + ' above 0'] = (same > 0).mean()
    blend_table[label + ' average'] = blend.mean(axis=0)
    blend_table[label + ' spread'] = blend.std(axis=0)
    blend_table[label + ' best in'] = np.bincount(blend.argmax(axis=1), minlength=len(blend_weights))

print("Correlation between the two halves (100 splits : average, spread from split to split, share of splits above 0)")
print(pd.DataFrame(same_table).round(3).to_string())
print()
print("Correlation of the blend with the high pressure value of the other half (200 = 100 splits, both directions)")
print(pd.DataFrame(blend_table, index=pd.Index(blend_weights, name='weight on level')).round(3).to_string())

Correlation between the two halves (100 splits : average, spread from split to split, share of splits above 0)
                     batters average  batters spread  batters above 0  bowlers average  bowlers spread  bowlers above 0
level                          0.318           0.089             1.00            0.201           0.094             0.98
high pressure value           -0.108           0.103             0.13            0.206           0.095             0.97
clutch                        -0.227           0.094             0.01            0.069           0.092             0.79
clutch (rule index)           -0.033           0.116             0.42            0.023           0.095             0.58
consistency                    0.017           0.093             0.55            0.255           0.088             1.00

Correlation of the blend with the high pressure value of the other half (200 = 100 splits, both directions)
                 batters average  batters spread  batters be

In [4]:
# Saving the two tables above for the charts of step 6
pd.DataFrame(same_table).round(4).to_csv(os.path.join(output_folder, "05_splithalf.csv"))
pd.DataFrame(blend_table, index=pd.Index(blend_weights, name='weight on level')).round(4).to_csv(os.path.join(output_folder, "05_blend.csv"))

In [5]:
# How far from 0 can the clutch correlation be by chance?
# Same trick as step 4 : inside every player I shuffle which of his balls are called high pressure, so nobody
# can have a clutch skill. Then the same split-half (20 splits per shuffle, 30 shuffles).
shuffler = np.random.RandomState(7)
for label, source, player, value in [
    ('batters', balls[balls['legal']], 'striker_id', 'batter_value'),
    ('bowlers', balls, 'bowler_id', 'bowler_value'),
]:
    players, eligible = len(groups[label]['ids']), groups[label]['eligible']
    by_chance = []
    for run in range(30):
        test = source.assign(high=source.groupby(player)['high'].transform(lambda x: shuffler.permutation(x.values)))
        rows, ids = match_rows(test, player, value)
        match_code = rows['match_code'].values
        found = []
        for split in range(20):
            first = (shuffler.rand(match_code.max() + 1) < 0.5)[match_code]
            halves = [player_numbers(rows[side], players)[eligible] for side in [first, ~first]]
            found.append(halves[0]['delta'].corr(halves[1]['delta']))
        by_chance.append(np.mean(found))
    real = same_table[label + ' average']['clutch']
    print(f"{label}: clutch correlation with shuffled labels, average {np.mean(by_chance):+.3f}, from {min(by_chance):+.3f} to {max(by_chance):+.3f}. "
          f"Real: {real:+.3f}, {np.mean(np.array(by_chance) <= real):.0%} of the shuffles are as low, {np.mean(np.array(by_chance) >= real):.0%} as high")

batters: clutch correlation with shuffled labels, average -0.002, from -0.246 to +0.222. Real: -0.227, 3% of the shuffles are as low, 97% as high
bowlers: clutch correlation with shuffled labels, average +0.008, from -0.246 to +0.236. Real: +0.069, 70% of the shuffles are as low, 30% as high


In [6]:
# Test 1c : the two parts of a value, runs and wickets
# A value is runs (scored or saved, against expected) plus wickets (lost or taken, in runs). Do the two come back
# equally in the other half of the matches? And what predicts the full value of the other half better :
# the level shrunk as one number, or shrunk as two parts and added? 100 splits, ranked players.
# For bowlers the two parts are what the ranking uses. For batters I only look, to see if they need the same.
balls['batter_runs_part'] = balls['bat_runs'] - balls['exp_bat_runs']
balls['batter_wicket_part'] = -balls['wicket_cost'] * (balls['striker_out'].astype(float) - balls['exp_bat_out'])
splitter = np.random.RandomState(9)
parts_table = {}
for label, source, part_values in [('batters', balls[balls['legal']], ['batter_runs_part', 'batter_wicket_part']),
                                   ('bowlers', balls, ['bowler_runs_part', 'bowler_wicket_part'])]:
    group = groups[label]
    rows, ids, in_pool, eligible, player = group['rows'], group['ids'], group['in_pool'], group['eligible'], group['player']
    two = [match_rows(source, player, part)[0] for part in part_values]
    match_code = rows['match_code'].values
    found = []
    for run in range(100):
        first = (splitter.rand(match_code.max() + 1) < 0.5)[match_code]
        for one, other in [(first, ~first), (~first, first)]:
            target = player_numbers(rows[other], len(ids))['raw_level'].values[eligible]
            line = {'runs part': np.nan, 'wicket part': np.nan}
            for name, part in zip(line, two):
                a, b = player_numbers(part[one], len(ids))['raw_level'][eligible], player_numbers(part[other], len(ids))['raw_level'][eligible]
                line[name] = a.corr(b)
            line['one number predicts other half'] = pd.Series(shrunk_parts([rows[one]], len(ids), in_pool)[eligible]).corr(pd.Series(target))
            line['two parts predict other half'] = pd.Series(shrunk_parts([part[one] for part in two], len(ids), in_pool)[eligible]).corr(pd.Series(target))
            found.append(line)
    found = pd.DataFrame(found)
    parts_table[label + ' average'] = found.mean()
    parts_table[label + ' spread'] = found.std()
    parts_table[label + ' two parts better in'] = (found['two parts predict other half'] > found['one number predicts other half']).mean()
    print(f"{label}: share of the variance of a ball value, runs {source[part_values[0]].var() / source[group['value']].var():.2f}, wickets {source[part_values[1]].var() / source[group['value']].var():.2f}")
parts_table = pd.DataFrame(parts_table)
parts_table.round(4).to_csv(os.path.join(output_folder, "05_parts.csv"))
print(parts_table.round(3).to_string())

batters: share of the variance of a ball value, runs 0.41, wickets 0.45
bowlers: share of the variance of a ball value, runs 0.41, wickets 0.45
                                batters average  batters spread  batters two parts better in  bowlers average  bowlers spread  bowlers two parts better in
runs part                                 0.619           0.068                         0.77            0.485           0.081                        0.915
wicket part                               0.372           0.100                         0.77            0.151           0.104                        0.915
one number predicts other half            0.355           0.087                         0.77            0.227           0.089                        0.915
two parts predict other half              0.382           0.088                         0.77            0.277           0.091                        0.915


In [7]:
# Test 2 : IPL against T20I
# The level again, from the balls of one competition only. Three questions :
# 1. Do the two competitions agree? Correlation of a player's IPL level with his T20I level (players with 300+ balls in both)
# 2. Is a T20I ball easier? The gap from the cell above, and what taking it off does to the step 4 top 10
# 3. Is the final list carried by T20I balls? The top 10 from IPL balls only against my top 10
for label, group in groups.items():
    rows, parts, ids, in_pool, eligible, mine = group['rows'], group['parts'], group['ids'], group['in_pool'], group['eligible'], group['mine']
    one_comp = {}
    for comp in ['IPL', 'T20I']:
        in_comp = (rows['comp_group'] == comp).values
        table = player_numbers(rows[in_comp], len(ids))
        table['level'] = shrunk_parts([part[in_comp] for part in parts], len(ids), in_pool & (table['balls'] >= 300).values)
        one_comp[comp] = table.set_index(ids)
    ipl, t20i = one_comp['IPL'], one_comp['T20I']
    both = (ipl['balls'] >= 300) & (t20i['balls'] >= 300)
    ranked_both = both & eligible
    print(f"{label}: {ranked_both.sum()} of the {eligible.sum()} ranked players have 300+ balls in both. "
          f"Correlation of IPL level with T20I level {ipl.loc[ranked_both, 'raw_level'].corr(t20i.loc[ranked_both, 'raw_level']):.2f} "
          f"(rank correlation {ipl.loc[ranked_both, 'raw_level'].corr(t20i.loc[ranked_both, 'raw_level'], method='spearman'):.2f})")

    gap, gaps = t20i_gap(rows, len(ids), in_pool)
    before = list(step4[label].index[:10])
    print(f"   the same player is {gap:+.3f} per ball better in T20Is ({len(gaps)} players, plain average {gaps.mean():+.3f}, give or take {gaps.std() / np.sqrt(len(gaps)):.3f}). "
          f"With that taken off every T20I ball, {len(set(before) & set(mine))} of the step 4 top 10 stay. "
          f"Out: {names[label][[p for p in before if p not in mine]].tolist()}. In: {names[label][[p for p in mine if p not in before]].tolist()}")
    level = group['level']
    print(f"   next 3 after my top 10: {names[label][list(level.index[10:13])].tolist()}, level of places 9 to 12: {level.iloc[8:12].round(3).tolist()}")

    ipl_only = list(ipl.loc[eligible, 'level'].sort_values(ascending=False).index[:10])
    print(f"   top 10 from IPL balls only: {len(set(mine) & set(ipl_only))} of my 10 stay. "
          f"Out: {names[label][[p for p in mine if p not in ipl_only]].tolist()}. In: {names[label][[p for p in ipl_only if p not in mine]].tolist()}")
    print(f"   rank correlation of the IPL only level with my final level, all ranked players: {ipl.loc[eligible, 'level'].corr(level, method='spearman'):.2f}")
    share = (ipl['balls'] / (ipl['balls'] + t20i['balls'])).loc[mine]
    print(f"   share of IPL balls in my top 10: {dict(zip(names[label][mine], share.round(2)))}")

batters: 37 of the 57 ranked players have 300+ balls in both. Correlation of IPL level with T20I level 0.30 (rank correlation 0.22)
   the same player is +0.084 per ball better in T20Is (40 players, plain average +0.082, give or take 0.026). With that taken off every T20I ball, 10 of the step 4 top 10 stay. Out: []. In: []
   next 3 after my top 10: ['Marcus Stoinis', 'Travis Head', 'Shivam Dube'], level of places 9 to 12: [0.083, 0.082, 0.08, 0.075]
   top 10 from IPL balls only: 7 of my 10 stay. Out: ['Phil Salt', 'Mitchell Marsh', 'Jos Buttler']. In: ['Shreyas Iyer', 'Travis Head', 'Shubman Gill']
   rank correlation of the IPL only level with my final level, all ranked players: 0.88
   share of IPL balls in my top 10: {'Abhishek Sharma': 0.54, 'Tim David': 0.53, 'Suryakumar Yadav': 0.49, 'Heinrich Klaasen': 0.85, 'Rajat Patidar': 1.0, 'Phil Salt': 0.44, 'Sai Sudharsan': 1.0, 'Mitchell Marsh': 0.61, 'Nicholas Pooran': 0.58, 'Jos Buttler': 0.63}
bowlers: 31 of the 62 ranked players h

In [8]:
# Test 3 : the rule index instead of my pressure score
# The lists are ranked on level, and the level does not use the pressure score at all. Pressure only decides
# who has enough high pressure balls and pressure matches to be ranked. So I pick the top fifth of balls with
# the simple rule index of step 2 (only 7% of all balls are high in both), and build the lists again.
for label, group in groups.items():
    rows, ids, eligible, mine = group['rows'], group['ids'], group['eligible'], group['mine']
    rule_rows = rows.assign(high_balls=rows['rule_balls'], high_total=rows['rule_total'])
    rule_pool, rule_eligible = who_is_ranked(rule_rows, len(ids), group['min_high'])
    rule_top = list(final_level(group['parts'], ids, rule_pool, rule_eligible, group['gaps']).index[:10])
    out = [p for p in mine if p not in rule_top]
    print(f"{label}: {rule_eligible.sum()} players ranked with the rule index ({(eligible & rule_eligible).sum()} of my {eligible.sum()}). "
          f"{len(set(mine) & set(rule_top))} of my top 10 stay. "
          f"Out: {names[label][out].tolist()}. In: {names[label][[p for p in rule_top if p not in mine]].tolist()}")
    print(f"   of those out, not ranked at all with the rule index (too few high balls or pressure matches): "
          f"{names[label][[p for p in out if p not in ids[rule_eligible]]].tolist()}")

    # The value on high pressure balls itself, under the two definitions
    in_both = eligible & rule_eligible
    print(f"   rank correlation of the raw high pressure value under the two definitions: "
          f"{group['table'].loc[in_both, 'raw_high'].corr(player_numbers(rule_rows, len(ids)).loc[in_both, 'raw_high'], method='spearman'):.2f}")

batters: 44 players ranked with the rule index (40 of my 57). 8 of my top 10 stay. Out: ['Abhishek Sharma', 'Rajat Patidar']. In: ['Andre Russell', 'Shreyas Iyer']
   of those out, not ranked at all with the rule index (too few high balls or pressure matches): ['Abhishek Sharma', 'Rajat Patidar']
   rank correlation of the raw high pressure value under the two definitions: 0.39
bowlers: 63 players ranked with the rule index (56 of my 62). 10 of my top 10 stay. Out: []. In: []
   of those out, not ranked at all with the rule index (too few high balls or pressure matches): []
   rank correlation of the raw high pressure value under the two definitions: 0.50


In [9]:
# Test 4 : other eligibility numbers
# My pick was 100 (batters) or 120 (bowlers) high pressure balls and 10 pressure matches, half of both in the IPL.
# The same lists with lower and higher minimums : how many players are ranked, and how many of my top 10 stay?
for label, group in groups.items():
    rows, ids, mine = group['rows'], group['ids'], group['mine']
    for min_high in [group['min_high'] * 0.6, group['min_high'], group['min_high'] * 1.5]:
        for min_matches in [5, 10, 15]:
            in_pool, eligible = who_is_ranked(rows, len(ids), min_high, min_matches)
            top = list(final_level(group['parts'], ids, in_pool, eligible, group['gaps']).index[:10])
            print(f"{label}: {int(min_high)}+ high pressure balls, {min_matches}+ pressure matches -> {eligible.sum()} ranked, "
                  f"{len(set(mine) & set(top))} of my top 10 stay. "
                  f"Out: {names[label][[p for p in mine if p not in top]].tolist()}. In: {names[label][[p for p in top if p not in mine]].tolist()}")

batters: 60+ high pressure balls, 5+ pressure matches -> 91 ranked, 8 of my top 10 stay. Out: ['Nicholas Pooran', 'Jos Buttler']. In: ['Vaibhav Sooryavanshi', 'Andre Russell']
batters: 60+ high pressure balls, 10+ pressure matches -> 57 ranked, 10 of my top 10 stay. Out: []. In: []
batters: 60+ high pressure balls, 15+ pressure matches -> 27 ranked, 7 of my top 10 stay. Out: ['Heinrich Klaasen', 'Rajat Patidar', 'Phil Salt']. In: ['Marcus Stoinis', 'Travis Head', 'Shreyas Iyer']
batters: 100+ high pressure balls, 5+ pressure matches -> 77 ranked, 8 of my top 10 stay. Out: ['Nicholas Pooran', 'Jos Buttler']. In: ['Vaibhav Sooryavanshi', 'Andre Russell']
batters: 100+ high pressure balls, 10+ pressure matches -> 57 ranked, 10 of my top 10 stay. Out: []. In: []
batters: 100+ high pressure balls, 15+ pressure matches -> 27 ranked, 7 of my top 10 stay. Out: ['Heinrich Klaasen', 'Rajat Patidar', 'Phil Salt']. In: ['Marcus Stoinis', 'Travis Head', 'Shivam Dube']
batters: 150+ high pressure ba

In [10]:
# Test 5 : choices of mine that move the numbers
# 1. The price of a wicket. A wicket is about half of the variance of a ball value (step 3), and the price (runs a wicket
#    costs) is my own estimate. I rank again with the price times 0, 0.5, 1.5 and 2. Times 0 = runs only.
#    (the T20I gap is measured again for every price, because the gap has a wicket part too)
# 2. The years. The value is an average over 2022 to 2026 with every year counted the same. I rank again on 2024 to 2026
#    and on 2025 to 2026 only (the pool for shrinking, the minimums and the T20I gap stay as they are).
# 3. No shrinking : the raw level of every ranked player, in IPL terms.
# 4. Bowlers only : shrunk as one number, the way the first version did it.
sensitivity = []
for label, group in groups.items():
    source, player, ids, in_pool, eligible, mine = group['source'], group['player'], group['ids'], group['in_pool'], group['eligible'], group['mine']
    scenarios = [('wicket price x 0 (runs only)', 0.0, 2022, 'parts'), ('wicket price x 0.5', 0.5, 2022, 'parts'), ('wicket price x 1.5', 1.5, 2022, 'parts'),
                 ('wicket price x 2', 2.0, 2022, 'parts'), ('years 2024 to 2026 only', 1.0, 2024, 'parts'), ('years 2025 to 2026 only', 1.0, 2025, 'parts'),
                 ('no shrinking (raw level)', 1.0, 2022, 'raw')]
    if label == 'bowlers':
        scenarios.append(('shrunk as one number', 1.0, 2022, 'one'))
    for name, price, first_year, how in scenarios:
        if label == 'batters':
            changed = [source['bat_runs'] - source['exp_bat_runs'] - price * source['wicket_cost'] * (source['striker_out'].astype(float) - source['exp_bat_out'])]
        else:
            changed = [source['bowler_runs_part'], price * source['bowler_wicket_part']]
            if how == 'one':
                changed = [changed[0] + changed[1]]
            elif price == 0:
                changed = changed[:1]
        parts = [match_rows(source.assign(changed=value), player, 'changed')[0] for value in changed]
        gaps = [t20i_gap(part, len(ids), in_pool)[0] for part in parts]
        recent = (parts[0]['match_id'].map(year_of_match) >= first_year).values
        parts = [part[recent] for part in parts]
        if how == 'raw':
            raw = sum(player_numbers(in_ipl_terms(part, gap), len(ids))['raw_level'].values for part, gap in zip(parts, gaps))
            level = pd.Series(raw, index=ids)[eligible].sort_values(ascending=False)
        else:
            level = final_level(parts, ids, in_pool, eligible, gaps)
        top = list(level.index[:10])
        sensitivity.append({'list': label, 'scenario': name, 'ranked': len(level), 'of_my_10_stay': len(set(top) & set(mine)),
                            'rank_correlation': level.corr(group['level'].reindex(level.index), method='spearman'),
                            'out': ", ".join(names[label][[p for p in mine if p not in top]]),
                            'in': ", ".join(names[label][[p for p in top if p not in mine]])})
sensitivity = pd.DataFrame(sensitivity)
sensitivity.round(3).to_csv(os.path.join(output_folder, "05_sensitivity.csv"), index=False)
pd.set_option('display.width', 250)
pd.set_option('display.max_colwidth', 90)
print(sensitivity.round(2).to_string(index=False))

# How often does a player of my final 10 drop out of the top 10 in the first 7 runs?
for label in ['batters', 'bowlers']:
    part = sensitivity[(sensitivity['list'] == label) & (sensitivity['scenario'] != 'shrunk as one number')]
    dropped = pd.Series(", ".join(part['out']).split(", ")).replace('', np.nan).dropna().value_counts()
    never = [name for name in names[label][groups[label]['mine']] if name not in dropped.index]
    print(f"{label}: times each player drops out in the 7 runs: {dropped.to_dict()}. Never: {never}")

   list                     scenario  ranked  of_my_10_stay  rank_correlation                                                       out                                                    in
batters wicket price x 0 (runs only)      57              7              0.67                Sai Sudharsan, Mitchell Marsh, Jos Buttler        Travis Head, Marcus Stoinis, Prabhsimran Singh
batters           wicket price x 0.5      57              8              0.90                                Sai Sudharsan, Jos Buttler                           Travis Head, Marcus Stoinis
batters           wicket price x 1.5      57              9              0.95                                           Nicholas Pooran                                           Virat Kohli
batters             wicket price x 2      57              7              0.88                Phil Salt, Mitchell Marsh, Nicholas Pooran               Virat Kohli, Lokesh Rahul, Shubman Gill
batters      years 2024 to 2026 only      57      

In [11]:
# Test 6 : looking forward
# Every test so far mixes the years. An auction buys the next season, so : what did the years up to 2025 say about 2026?
# I build the level (in IPL terms) from the earlier years only and compare it with the raw level in 2026, also in IPL terms
# (ranked players with 150+ balls in 2026). Three versions : all years up to 2025, 2024 and 2025, and 2025 only.
# One season and about 40 players is a small test : a correlation here moves by about 0.15 on luck alone.
forward = []
for label, group in groups.items():
    parts, ids, in_pool, eligible = group['ipl_parts'], group['ids'], group['in_pool'], group['eligible']
    year = parts[0]['match_id'].map(year_of_match).values
    in_2026 = [player_numbers(part[year == 2026], len(ids)) for part in parts]
    after = pd.Series(sum(table['raw_level'].values for table in in_2026), index=ids)[in_2026[0]['balls'].values >= 150]
    for name, first_year in [('2022 to 2025', 2022), ('2024 to 2025', 2024), ('2025 only', 2025)]:
        earlier = (year >= first_year) & (year <= 2025)
        before = final_level([part[earlier] for part in parts], ids, in_pool, eligible, [0.0] * len(parts))
        both = pd.concat([before.rename('before'), after.rename('after')], axis=1).dropna()
        forward.append({'list': label, 'level built on': name, 'players': len(both),
                        'correlation with 2026': both['before'].corr(both['after']),
                        'rank correlation': both['before'].corr(both['after'], method='spearman')})
forward = pd.DataFrame(forward)
forward.round(3).to_csv(os.path.join(output_folder, "05_forward.csv"), index=False)
print(forward.round(2).to_string(index=False))

   list level built on  players  correlation with 2026  rank correlation
batters   2022 to 2025       40                   0.04              0.00
batters   2024 to 2025       40                   0.11             -0.02
batters      2025 only       40                   0.24              0.12
bowlers   2022 to 2025       42                   0.26              0.30
bowlers   2024 to 2025       42                  -0.01             -0.02
bowlers      2025 only       40                   0.05              0.12


In [13]:
# Final lists : ranked on the level in IPL terms (the T20I gap taken off every T20I ball), because the lists
# are for an IPL auction.
#
# Rank confidence : the matches I have are one draw of the matches that could have been played. I draw the same
# number of matches again with repeats (some matches twice, some not at all), 1,000 times, and rank the players
# again every time. The ranked players, the pool, the T20I gap and how hard every player is pulled towards the
# average stay the same, so this shows only how far luck in the matches moves a rank.
# (The pull has to stay fixed : drawing matches again adds luck to every raw level, the spread between players
# then looks bigger than it is, and a pull measured again inside every draw would be too weak.)
#   top10_share : in how many of the 1,000 draws he is in the top 10
#   place_low / place_high : 90% of the draws put him between these two places
drawer = np.random.RandomState(11)
final = {}
for label, group in groups.items():
    parts, ids, in_pool, eligible = group['ipl_parts'], group['ids'], group['in_pool'], group['eligible']
    # For every part : the pool average and the pull of every player, from the real matches.
    # shrunk level = centre + (raw - centre) * pull, so pull = (shrunk - centre) / (raw - centre)
    centres, pulls = [], []
    for rows in parts:
        table = player_numbers(rows, len(ids))
        raw = table['raw_level'].values
        centre = raw[in_pool].mean()
        centres.append(centre)
        pulls.append(np.nan_to_num((shrunk_level(rows, table, in_pool) - centre) / (raw - centre)))

    match_code = parts[0]['match_code'].values
    matches = match_code.max() + 1
    places = np.zeros((1000, eligible.sum()), dtype=int)
    for run in range(1000):
        weight = drawer.multinomial(matches, np.ones(matches) / matches)[match_code].astype(float)
        level = sum(centre + (player_numbers(rows, len(ids), weight)['raw_level'].values - centre) * pull
                    for rows, centre, pull in zip(parts, centres, pulls))[eligible]
        places[run] = (-level).argsort().argsort() + 1

    table = step4[label].rename(columns={'rank': 'step4_rank'}).join(pd.DataFrame({
        'top10_share': (places <= 10).mean(axis=0),
        'place_median': np.median(places, axis=0),
        'place_low': np.percentile(places, 5, axis=0),
        'place_high': np.percentile(places, 95, axis=0),
    }, index=ids[eligible])).join(group['level'].rename('ipl_level'))
    table = table.sort_values('ipl_level', ascending=False)
    table.insert(0, 'rank', np.arange(1, len(table) + 1))
    table.round(4).to_csv(os.path.join(output_folder, f"05_{label}.csv"))
    final[label] = table

    top, rest = table.head(10), table.iloc[10:]
    print(f"{label}: {int((top['top10_share'] >= 0.5).sum())} of the top 10 are in the top 10 in at least half of the draws. "
          f"Outside the top 10 with a share of 25% or more: {rest[rest['top10_share'] >= 0.25]['name'].tolist()}")
    print(table[['rank', 'step4_rank', 'name', 'balls', 'level', 'ipl_level', 'top10_share', 'place_median', 'place_low', 'place_high']].head(15).round(3).to_string(index=False))
    print()

batters: 7 of the top 10 are in the top 10 in at least half of the draws. Outside the top 10 with a share of 25% or more: ['Marcus Stoinis', 'Travis Head']
 rank  step4_rank             name  balls  level  ipl_level  top10_share  place_median  place_low  place_high
    1           1  Abhishek Sharma   2239  0.261      0.229        0.969           1.0        1.0        8.00
    2           2        Tim David   1210  0.226      0.195        0.969           3.0        1.0        9.00
    3           3 Suryakumar Yadav   2746  0.219      0.181        0.978           3.0        1.0        9.00
    4           4 Heinrich Klaasen   1408  0.178      0.161        0.948           4.0        1.0       11.00
    5           6    Rajat Patidar    916  0.163      0.152        0.886           5.0        2.0       14.00
    6           5        Phil Salt   1628  0.166      0.126        0.675           7.0        2.0       32.05
    7          10    Sai Sudharsan   1674  0.110      0.100        0.516  

### What step 5 found

- Level comes back in the other half, clutch does not. Level: 0.32 for batters and 0.20 for bowlers, above 0 in almost every split. Clutch: -0.23 for batters and +0.07 for bowlers. With shuffled labels (no clutch possible) the same number runs from about -0.25 to +0.24, so neither is a skill I can see.
- Level is the number to rank on. For batters it predicts the high pressure value of the other half at 0.14, against -0.11 for the high pressure balls themselves, and it is the best weight in 197 of 200 tries. For bowlers level (0.22), high pressure balls only (0.21) and the best blend (0.24) are all inside the spread of 0.10.
- A bowler's runs saved come back in the other half at 0.49, his wickets at 0.15. Shrunk in two parts, the bowler level predicts the other half at 0.28 against 0.23 as one number, and it is better in 92% of 200 tries. So bowlers are ranked on the two parts. Batters would gain less (0.38 against 0.35) and stay as one number.
- T20I balls are easier: the same player is 0.08 (batters) and 0.04 (bowlers) runs per ball better in T20Is. The final lists take that off every T20I ball. Both top 10s keep their names, only the order moves.
- With the rule index all 10 bowlers stay and 8 batters stay (Abhishek Sharma and Patidar have fewer than 10 pressure matches under it).
- The ball minimum hardly matters. The match minimum does: with 5 matches Sooryavanshi and Russell come in for the batters, with 15 Klaasen, Patidar and Salt (batters) and Ellis and Santner (bowlers) are no longer ranked.
- My own choices (wicket price, seasons, no shrinking) move 0 to 4 names. Abhishek Sharma, Tim David, Klaasen and Patidar (batters) and Bumrah, Narine and Ellis (bowlers) never drop out. Buttler and Pooran drop out in 5 of 7 runs, Santner in 4.
- Looking forward is weak: the level from 2022 to 2025 predicts 2026 at 0.04 for batters and 0.26 for bowlers, and no shorter window is better for both. One season of about 40 players is very noisy.
- The top of both final lists is firm and the last places are close. Batters 1 to 5 are in the top 10 in 89% to 98% of the draws, places 7 to 10 in 35% to 52%. Bowlers 1 to 3 in 85% to 100%, places 8 to 10 in 37% to 40%. Head and Stoinis (batters) and Mohsin Khan (bowlers) are the closest players outside, at about 35%.